# Transfer learning

MichAl Academy, unit 3.13.

Three questions, all on Fashion-MNIST, all with everything held identical except
the one thing being compared:

1. Does starting from a trained network beat starting from random weights?
2. Should the reused layers be frozen or allowed to keep learning?
3. Does it matter what the reused network was originally trained on?

**This notebook runs fewer seeds than the lesson does.** The lesson's tables come
from nine label draws and its paired comparison from twenty-one; here it is three
and nine, so that the whole thing finishes in a couple of minutes. The numbers
will be close but not identical, and the direction of every result is the same.


In [ ]:
import copy
import time
import warnings

import numpy as np
import torch
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

warnings.filterwarnings("ignore")
torch.set_num_threads(1)

NAMES = ["T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
         "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"]

data = fetch_openml("Fashion-MNIST", version=1, as_frame=False)
images = data.data.astype("float32") / 255.0
labels = data.target.astype(int)

print(f"{len(images)} photographs in ten categories")


## The target task

Pullover against shirt, the pair a plain network confuses most often. Two
categories, so chance is 0.50.

The source networks below are trained on the *other* categories and never see a
pullover or a shirt.


In [ ]:
target = np.isin(labels, [2, 6])
X_target = images[target]
y_target = (labels[target] == 6).astype(int)

X_pool, X_held, y_pool, y_held = train_test_split(
    X_target, y_target, test_size=0.3, stratify=y_target, random_state=0)

X_held_t = torch.tensor(X_held)
y_held_t = torch.tensor(y_held)

print(f"{len(y_pool)} to draw labels from, {len(y_held)} held back")


In [ ]:
def body():
    """The shared shape: 784 -> 128 -> 64, ReLU throughout."""
    return [nn.Linear(784, 128), nn.ReLU(), nn.Linear(128, 64), nn.ReLU()]


def run(model, X, y, steps, seed, lr=0.001, batch_size=16):
    """Train for exactly `steps` weight updates. Frozen layers are skipped."""
    torch.manual_seed(seed)
    trainable = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.Adam(trainable, lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    loader = DataLoader(TensorDataset(torch.tensor(X), torch.tensor(y)),
                        batch_size=batch_size, shuffle=True)
    done = 0
    while done < steps:
        for batch_X, batch_y in loader:
            opt.zero_grad()
            loss_fn(model(batch_X), batch_y).backward()
            opt.step()
            done += 1
            if done >= steps:
                break
    return model


def held_back_accuracy(model):
    model.eval()
    with torch.no_grad():
        out = model(X_held_t).argmax(dim=1)
    model.train()
    return float((out == y_held_t).float().mean())


def draw_labels(n, seed):
    """One draw: n labels, balanced between the two categories."""
    rng = np.random.default_rng(seed)
    picked = np.concatenate([
        rng.choice(np.flatnonzero(y_pool == c), n // 2, replace=False)
        for c in (0, 1)])
    return X_pool[picked], y_pool[picked]


## Building a pretrained network

There is no download here. The source network is trained in this notebook on
categories the target task never uses, which is the only way to be certain it
has never seen a pullover or a shirt.


In [ ]:
def pretrain(categories, rows=8000, seed=0):
    renumber = {c: i for i, c in enumerate(categories)}
    keep = np.isin(labels, categories)
    X_src = images[keep]
    y_src = np.array([renumber[v] for v in labels[keep]])

    rng = np.random.default_rng(7)
    take = rng.choice(len(y_src), rows, replace=False)

    torch.manual_seed(seed)
    net = nn.Sequential(*body(), nn.Linear(64, len(categories)))
    return run(net, X_src[take], y_src[take], steps=3000, seed=seed, batch_size=64)


started = time.time()
EIGHT = [0, 1, 3, 4, 5, 7, 8, 9]
source_net = pretrain(EIGHT, rows=24000)
print(f"trained on {len(EIGHT)} categories, none of them pullover or shirt "
      f"({time.time() - started:.1f}s)")


## Question 1 and 2: random start, frozen, or fine-tuned

Three starting points, one identical training budget: 300 weight updates, Adam,
learning rate 0.001, batch size 16, on exactly the same drawn labels.

**The budget has to be identical.** An earlier version of this measurement gave
the fine-tuned network a learning rate three times smaller, on the reasoning that
a fine-tune should tread carefully, and it lost at every size. That was the
handicap talking, not the initialisation.


In [ ]:
STEPS, SEEDS = 300, 3

print("labels   random   frozen   fine-tuned")
for n in (10, 50, 250, 1000):
    scores = [[], [], []]
    for seed in range(SEEDS):
        X_few, y_few = draw_labels(n, seed)

        torch.manual_seed(seed)
        scratch = nn.Sequential(*body(), nn.Linear(64, 2))
        scores[0].append(held_back_accuracy(run(scratch, X_few, y_few, STEPS, seed)))

        frozen = copy.deepcopy(source_net)
        torch.manual_seed(seed)
        frozen[4] = nn.Linear(64, 2)
        for p in list(frozen[0].parameters()) + list(frozen[2].parameters()):
            p.requires_grad = False
        scores[1].append(held_back_accuracy(run(frozen, X_few, y_few, STEPS, seed)))

        tuned = copy.deepcopy(source_net)
        torch.manual_seed(seed)
        tuned[4] = nn.Linear(64, 2)
        scores[2].append(held_back_accuracy(run(tuned, X_few, y_few, STEPS, seed)))

    print(f"{n:6d}   {np.median(scores[0]):.4f}   {np.median(scores[1]):.4f}"
          f"   {np.median(scores[2]):.4f}")


Fine-tuning is ahead of a random start, and freezing is behind it.

Freezing loses because the frozen 64 numbers were optimised to separate trousers
from sandals from bags. Nothing in that task needed the difference between
knitted and woven fabric, so those numbers need not carry it, and no output layer
can recover what is not there. Fine-tuning can change what the body measures;
freezing cannot.

Freezing is still the option that exists when a network is too large to fine-tune
on the hardware you have, and for a network trained on a thousand categories
rather than eight it costs much less than it does here.


## Question 3: does the source task matter?

Two source networks, identical in architecture, rows and training steps. The only
difference is which four categories they were trained on.

- **Other garments**: T-shirt, dress, coat, trouser.
- **Footwear and bags**: sandal, sneaker, ankle boot, bag.

Neither has seen a pullover or a shirt. Each draw trains all three starting points
on exactly the same labels, so the comparison is paired.


In [ ]:
started = time.time()
garments = pretrain([0, 3, 4, 1], rows=8000)
footwear = pretrain([5, 7, 9, 8], rows=8000)
print(f"two source networks ({time.time() - started:.1f}s)\n")

DRAWS = 9
random_start, from_garments, from_footwear = [], [], []

for seed in range(DRAWS):
    X_few, y_few = draw_labels(50, seed)

    torch.manual_seed(seed)
    scratch = nn.Sequential(*body(), nn.Linear(64, 2))
    random_start.append(held_back_accuracy(run(scratch, X_few, y_few, STEPS, seed)))

    for source, store in ((garments, from_garments), (footwear, from_footwear)):
        net = copy.deepcopy(source)
        torch.manual_seed(seed)
        net[4] = nn.Linear(64, 2)
        store.append(held_back_accuracy(run(net, X_few, y_few, STEPS, seed)))

random_start = np.array(random_start)
from_garments = np.array(from_garments)
from_footwear = np.array(from_footwear)

print(f"random start     median {np.median(random_start):.4f}")
print(f"garment source   median {np.median(from_garments):.4f}")
print(f"footwear source  median {np.median(from_footwear):.4f}\n")
print(f"garments beat a random start in {int((from_garments > random_start).sum())}/{DRAWS} draws"
      f"   mean {float((from_garments - random_start).mean()):+.4f}")
print(f"footwear beat a random start in {int((from_footwear > random_start).sum())}/{DRAWS} draws"
      f"   mean {float((from_footwear - random_start).mean()):+.4f}")


**The garment source helps and the footwear source hurts.** That is negative
transfer: a pretrained start that leaves you worse off than random weights.

Random weights are unopinionated. A network trained to separate sandals from
boots is opinionated by construction: it learned to attend to soles and outlines
and to throw away the fabric texture that was useless for its own task. Three
hundred steps on fifty labels is not enough to undo that, so it is not that the
footwear network knows nothing useful. It is that it knows something wrong.

Count draws rather than comparing medians. Run this notebook twice and the
medians move; the win counts move much less, because each draw trains all three
starting points on exactly the same labels.


## What to take away

- Start from a pretrained network when one exists, and always when labels are
  scarce.
- Fine-tune by default. Freeze when the network is too large to fine-tune, or
  when it is enormous relative to your task.
- Check what the source was trained to tell apart before trusting it, and
  measure it against random weights rather than assuming it helps.

The networks used for this in practice are downloaded rather than trained, and
are about a hundred times the size of the ones here. That is lesson 3.13.4, and
it is where Track 4 picks up.
